In [ ]:
import os, json, math
from collections import Counter
from PIL import Image
import matplotlib.pyplot as plt

In [ ]:
# pip install pycocotools matplotlib pillow
try:
    from pycocotools.coco import COCO
except Exception as e:
    print("Install pycocotools: pip install pycocotools")
    raise

In [ ]:
coco_ann_file = "/path/to/annotations/captions_train2017.json"   # e.g., COCO train captions json
coco_img_dir   = "/path/to/train2017/"                         # directory with images

coco = COCO(coco_ann_file)

# Basic stats
ann_ids = coco.getAnnIds()
anns = coco.loadAnns(ann_ids)
img_ids = coco.getImgIds()
imgs = coco.loadImgs(img_ids)

print("Total annotations:", len(anns))
print("Total images:", len(imgs))

# Caption lengths
lengths = [len(a['caption'].split()) for a in anns]
import numpy as np
print("Caption length: mean {:.2f}, median {}, max {}".format(np.mean(lengths), np.median(lengths), np.max(lengths)))

# Compute distribution of image resolutions
res_counter = Counter((img['width'], img['height']) for img in imgs)
most_common_res = res_counter.most_common(10)
print("Most common resolutions (top10):", most_common_res[:10])

# Number of captions per image (should be 5 for COCO)
captions_per_image = Counter()
for a in anns:
    captions_per_image[a['image_id']] += 1
print("Caption counts distribution (sample):", list(captions_per_image.items())[:5])
print("Unique images with captions:", len(captions_per_image))

# Show several images with their captions
def show_image_with_captions(img_dict, anns_for_img, img_dir=coco_img_dir):
    img_path = os.path.join(img_dir, img_dict['file_name'])
    im = Image.open(img_path).convert('RGB')
    plt.figure(figsize=(8,6))
    plt.imshow(im); plt.axis('off')
    plt.title("\n".join([a['caption'] for a in anns_for_img]), fontsize=10)
    plt.show()

# pick 3 random images
import random
random.seed(42)
sample_img_ids = random.sample(img_ids, 3)
for iid in sample_img_ids:
    img = coco.loadImgs(iid)[0]
    ann_ids = coco.getAnnIds(imgIds=iid)
    anns_img = coco.loadAnns(ann_ids)
    show_image_with_captions(img, anns_img)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm
import os, json

# Option A: Tokenize + encode (useful if your generator uses token-level cross-attention)
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased", use_fast=True)
def encode_texts_tokenized(texts, max_length=32):
    # returns dict of input_ids, attention_mask tensors
    enc = tokenizer(texts, padding='max_length', truncation=True, max_length=max_length, return_tensors="pt")
    return enc  # enc['input_ids'], enc['attention_mask']

# Example:
texts = ["a red flower", "a cat sitting on a mat"]
enc = encode_texts_tokenized(texts)
print(enc['input_ids'].shape)  # (batch, max_length)

# Option B: Sentence / semantic embeddings (CLIP or sentence-transformers)
# Use CLIP text encoder (better aligned with images) OR sentence-transformers for semantic embeddings.

# CLIP (recommended for image-text alignment)
from transformers import CLIPTokenizer, CLIPTextModel
clip_tokenizer = CLIPTokenizer.from_pretrained("openai/clip-vit-base-patch32")
clip_text_encoder = CLIPTextModel.from_pretrained("openai/clip-vit-base-patch32")

def clip_text_embeddings(texts, device='cuda'):
    inputs = clip_tokenizer(texts, padding=True, truncation=True, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = clip_text_encoder(**inputs)
        # pooled_output or last_hidden_state:
        # use last_hidden_state for token-level cross-attention, pooled_output for single vector
        token_embeds = outputs.last_hidden_state  # (batch, seq_len, dim)
        pooled = outputs.pooler_output            # (batch, dim)  [if available]
        # For CLIPTextModel pooler_output may be None; instead use mean or use token 0.
        # We normalize pooled vector:
        pooled = token_embeds.mean(dim=1)
        pooled = pooled / pooled.norm(dim=-1, keepdim=True)
    return token_embeds, pooled

# Example:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
texts = ["a dog running", "a bowl of fruit on a table"]
token_emb, pooled_emb = clip_text_embeddings(texts, device=device)
print("token_emb", token_emb.shape, "pooled", pooled_emb.shape)

# Save embeddings to disk for faster training
import numpy as np
def save_embeddings(pooled_emb, fname="text_embeddings.npy"):
    np.save(fname, pooled_emb.cpu().numpy())

save_embeddings(pooled_emb, "sample_emb.npy")

In [ ]:
import os, math, random
from PIL import Image, ImageDraw
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision.utils import make_grid, save_image

# --- Dataset: generate shapes on the fly and cache ---
SHAPES = ["circle","square","triangle"]
IMG_SIZE = 64

def draw_shape(shape, size=IMG_SIZE, color=(255,255,255), bg=(0,0,0)):
    img = Image.new("RGB", (size,size), bg)
    draw = ImageDraw.Draw(img)
    padding = int(size*0.15)
    if shape == "circle":
        draw.ellipse([padding,padding,size-padding,size-padding], fill=color)
    elif shape == "square":
        draw.rectangle([padding,padding,size-padding,size-padding], fill=color)
    elif shape == "triangle":
        draw.polygon([(size//2,padding),(size-padding,size-padding),(padding,size-padding)], fill=color)
    return img

class ShapesDataset(Dataset):
    def __init__(self, n=5000, transform=None):
        self.n = n
        self.transform = transform
        self.items = [random.choice(SHAPES) for _ in range(n)]
    def __len__(self): return self.n
    def __getitem__(self, i):
        shape = self.items[i]
        img = draw_shape(shape)
        if self.transform:
            img = self.transform(img)
        label = SHAPES.index(shape)
        return img, label

# transforms
import torchvision.transforms as T
transform = T.Compose([
    T.ToTensor(),           # [0,1]
    T.Normalize([0.5]*3, [0.5]*3)  # [-1,1]
])

# --- Simple conditional GAN ---
z_dim = 128
num_classes = len(SHAPES)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

class Generator(nn.Module):
    def __init__(self, z_dim, embed_dim=50, ngf=128):
        super().__init__()
        self.label_emb = nn.Embedding(num_classes, embed_dim)
        self.net = nn.Sequential(
            nn.Linear(z_dim + embed_dim, ngf*8*4*4),
            nn.ReLU(True),
            nn.Unflatten(1, (ngf*8, 4, 4)),
            nn.ConvTranspose2d(ngf*8, ngf*4, 4, 2, 1), nn.BatchNorm2d(ngf*4), nn.ReLU(True),
            nn.ConvTranspose2d(ngf*4, ngf*2, 4, 2, 1), nn.BatchNorm2d(ngf*2), nn.ReLU(True),
            nn.ConvTranspose2d(ngf*2, ngf, 4, 2, 1), nn.BatchNorm2d(ngf), nn.ReLU(True),
            nn.ConvTranspose2d(ngf, 3, 3, 1, 1), nn.Tanh()
        )
    def forward(self, z, labels):
        le = self.label_emb(labels)
        x = torch.cat([z, le], dim=1)
        return self.net(x)

class Discriminator(nn.Module):
    def __init__(self, embed_dim=50, ndf=64):
        super().__init__()
        self.label_emb = nn.Embedding(num_classes, embed_dim)
        self.img_net = nn.Sequential(
            nn.Conv2d(3, ndf, 4, 2, 1), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf, ndf*2, 4, 2, 1), nn.BatchNorm2d(ndf*2), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf*2, ndf*4, 4, 2, 1), nn.BatchNorm2d(ndf*4), nn.LeakyReLU(0.2, inplace=True),
            nn.AdaptiveAvgPool2d(1)
        )
        self.fc = nn.Linear(ndf*4 + embed_dim, 1)
    def forward(self, img, labels):
        feat = self.img_net(img).view(img.size(0), -1)
        le = self.label_emb(labels)
        x = torch.cat([feat, le], dim=1)
        return torch.sigmoid(self.fc(x))

# instantiate
G = Generator(z_dim).to(device)
D = Discriminator().to(device)
optG = optim.Adam(G.parameters(), lr=2e-4, betas=(0.5,0.999))
optD = optim.Adam(D.parameters(), lr=2e-4, betas=(0.5,0.999))
bce = nn.BCELoss()

# dataloader
ds = ShapesDataset(n=4000, transform=transform)
dl = DataLoader(ds, batch_size=64, shuffle=True, num_workers=2)

# training loop (small)
epochs = 10
fixed_z = torch.randn(9, z_dim, device=device)
fixed_labels = torch.tensor([0,1,2]*3, device=device)  # example grid: circle,square,triangle

for epoch in range(epochs):
    for imgs, labels in dl:
        imgs = imgs.to(device)
        labels = labels.to(device)
        bs = imgs.size(0)

        # train D
        optD.zero_grad()
        real_y = torch.ones(bs,1, device=device)
        fake_y = torch.zeros(bs,1, device=device)
        out_real = D(imgs, labels)
        lossD_real = bce(out_real, real_y)

        z = torch.randn(bs, z_dim, device=device)
        gen_imgs = G(z, labels)
        out_fake = D(gen_imgs.detach(), labels)
        lossD_fake = bce(out_fake, fake_y)

        lossD = (lossD_real + lossD_fake) * 0.5
        lossD.backward(); optD.step()

        # train G
        optG.zero_grad()
        out_fake2 = D(gen_imgs, labels)
        lossG = bce(out_fake2, real_y)
        lossG.backward(); optG.step()

    print(f"Epoch {epoch+1}/{epochs} — D: {lossD.item():.4f} G: {lossG.item():.4f}")

    # save sample grid
    with torch.no_grad():
        sample = G(fixed_z, fixed_labels).cpu()
        save_image((sample+1)/2, f"samples_epoch_{epoch+1}.png", nrow=3)


In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F

class SelfAttention(nn.Module):
    def __init__(self, dim, heads=8):
        super().__init__()
        self.mha = nn.MultiheadAttention(embed_dim=dim, num_heads=heads, batch_first=True)
        self.ln = nn.LayerNorm(dim)
    def forward(self, x):
        # x: B,C,H,W -> (B, N, C)
        B,C,H,W = x.shape
        xflat = x.view(B, C, -1).permute(0,2,1)
        attn_out,_ = self.mha(xflat, xflat, xflat)
        out = self.ln(xflat + attn_out)
        return out.permute(0,2,1).view(B,C,H,W)

class CrossAttentionBlock(nn.Module):
    def __init__(self, feat_dim, text_dim, heads=8):
        super().__init__()
        self.q_proj = nn.Conv2d(feat_dim, feat_dim, 1)
        self.k_proj = nn.Linear(text_dim, feat_dim)
        self.v_proj = nn.Linear(text_dim, feat_dim)
        self.mha = nn.MultiheadAttention(embed_dim=feat_dim, num_heads=heads, batch_first=True)
        self.ln = nn.LayerNorm(feat_dim)
    def forward(self, feat_map, text_tokens):
        B,C,H,W = feat_map.shape
        q = self.q_proj(feat_map).view(B, C, -1).permute(0,2,1)  # (B, N, C)
        k = self.k_proj(text_tokens)  # (B, T, C)
        v = self.v_proj(text_tokens)
        attn_out, _ = self.mha(q, k, v)
        out = self.ln(q + attn_out)
        return out.permute(0,2,1).view(B,C,H,W)

Generated images shape: torch.Size([4, 3, 32, 32])
Discriminator output shape: torch.Size([4, 1])


In [ ]:
import torch
from PIL import Image
from transformers import CLIPProcessor, CLIPModel
from pathlib import Path

model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").eval().to('cuda')
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

def clip_score(image_paths, captions):
    imgs = [Image.open(p).convert('RGB') for p in image_paths]
    inputs = processor(text=captions, images=imgs, return_tensors="pt", padding=True).to('cuda')
    outputs = model(**inputs)
    image_emb = outputs.image_embeds / outputs.image_embeds.norm(dim=-1, keepdim=True)
    text_emb = outputs.text_embeds / outputs.text_embeds.norm(dim=-1, keepdim=True)
    scores = (image_emb * text_emb).sum(dim=-1).cpu().numpy()
    return scores